# Video2Frames Prompt Tuning (APO) — reward v1

本 notebook 按 [README.md](README.md) 的流程，把每个运行步骤整理成可以逐个执行的 cell。

任务背景：用 Agent-Lightning 的 APO（自动提示词优化）算法，调优视频监控帧分析任务的固定指令 prompt。
客户数据集 `original_data/qwen_0318_swift_task.json` 含 5850 条视频分析任务（视频 → 结构化 JSON：`english_detail` / `brief` / `title` / `scene_type` / `is_courier_action`）。
每条视频任务被转换为**多帧图片任务**：去掉 `<video>` 占位符，在指令后追加逐帧占位符：

```
<frame 1 | 0s> <frame 2 | 3s> ... <frame n | 3(n-1)s>
```

APO 只调优 prompt 的**固定指令部分**；帧占位符部分在运行时按任务重建。

> **重要：** `original_data/qwen_0318_swift_task.json` 是客户数据，绝不能提交或推送到 GitHub。
> `original_data/`、`data/`、`log/`、`results/` 目录内容已被 git 忽略。
> 请单独（如 scp）把 `original_data/`、`data/` 和仓库根目录的 `.env` 拷到训练机器上。

## 0. 工作目录

确保后续所有命令都在 `video2frames-prompt-tuning/` 目录下执行。

In [ ]:
import os

# 若 notebook 从仓库根目录打开，则切换到项目目录
if os.path.basename(os.getcwd()) != "video2frames-prompt-tuning":
    os.chdir("video2frames-prompt-tuning")
print("当前工作目录:", os.getcwd())

# 强制用 .env 覆盖内核继承的环境变量。
# Jupyter server 启动时环境里可能残留旧的 AZURE_OPENAI_* / OPENAI_API_VERSION，
# 而代码内部的 load_dotenv(override=False) 不会覆盖它们，会导致 404 等诡异错误；
# 这里 override=True 保证后续 ! 子进程拿到的是 .env 里的值。
from dotenv import load_dotenv

load_dotenv("../.env", override=True)
load_dotenv(".env", override=True)
if not os.environ.get("OPENAI_API_VERSION") and os.environ.get("AZURE_OPENAI_API_VERSION"):
    os.environ["OPENAI_API_VERSION"] = os.environ["AZURE_OPENAI_API_VERSION"]
for key in ("AZURE_OPENAI_ENDPOINT", "OPENAI_API_VERSION", "AZURE_OPENAI_DEPLOYMENT"):
    print(key, "=", os.environ.get(key))

## 1. 安装

本项目必须使用**从本仓库源码构建的 agent-lightning 0.3.1**（PyPI 上的 0.3.0 缺少必需功能）。

In [ ]:
!python -m venv .venv
!.venv/bin/pip install -r requirements.txt

In [ ]:
# 期望输出 0.3.1
!.venv/bin/python -c "import agentlightning; print(agentlightning.__version__)"

## 2. 配置

Blob 存储配置从仓库根目录 `.env` 读取（`blob4videodatasets_connection_string`、
`blob4videodatasets_container_name`、`blob4videodatasets_frames_folder_name`）。
此外还需在 `.env` 中添加（或 export）以下 Azure OpenAI 变量：

| 变量 | 用途 | 默认值 |
| --- | --- | --- |
| `AZURE_OPENAI_ENDPOINT` | Azure OpenAI endpoint URL | 必填 |
| `AZURE_OPENAI_API_KEY` | Azure OpenAI API key | 必填 |
| `OPENAI_API_VERSION` | Azure OpenAI API 版本（如 `2024-10-21`） | 必填 |
| `AZURE_OPENAI_DEPLOYMENT` | 分析帧所用的多模态 deployment | `gpt-4o` |
| `JUDGE_MODEL` | reward 中 LLM judge 使用的 deployment | `gpt-4.1-mini` |
| `APO_GRADIENT_MODEL` | APO 批评（critique）prompt 用的 deployment | `gpt-4.1` |
| `APO_APPLY_EDIT_MODEL` | APO 重写 prompt 用的 deployment | `gpt-4.1-mini` |
| `FRAMES_AS_BASE64` | 设为 `true` 时以 base64 data URI 发送帧（而非 SAS URL） | 未设置 |

In [ ]:
# 快速检查 .env 中的关键变量是否已配置（不打印值）
from pathlib import Path

env_path = Path("..") / ".env"
required = [
    "blob4videodatasets_connection_string",
    "blob4videodatasets_container_name",
    "blob4videodatasets_frames_folder_name",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "OPENAI_API_VERSION",
]
if env_path.exists():
    keys = {line.split("=", 1)[0].strip() for line in env_path.read_text().splitlines()
            if "=" in line and not line.strip().startswith("#")}
    for name in required:
        print(f"{'OK  ' if name in keys else 'MISS'} {name}")
else:
    print(f"未找到 {env_path.resolve()} — 请先准备 .env")

## 3. 准备数据集（Workflow 步骤 1）

分层抽样并从 Azure 解析帧 blob：

- 抽样与切分按 (family, `is_courier_action`) 联合分层，每个 split 都镜像候选池的标签分布；
  val split 保证至少 `--val-courier-min`（默认 0.15）的快递员正例，并记录各 split 的
  courier / scene_type / family 分布（scene_type 只报告分布，无配额）。
- `--probe-content-filter` 在抽样时用 Azure 内容安全过滤器探测每个候选视频
  （约 3% 的视频无论 prompt 如何都会被拒绝），被拦截的会回填替换，保证各 split 达到目标大小。
  探测结果按视频缓存在 `data/content_filter_cache.json`，重复运行不会重复探测。

In [ ]:
# 输出很长（azure blob 的每个请求都打 INFO 日志），重定向到文件，只看结尾摘要
!mkdir -p log
!.venv/bin/python prepare_data.py --train-size 40 --val-size 24 --test-size 30 --seed 42 --probe-content-filter \
    > log/prepare_data.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/prepare_data.log

### 3b.（第二轮）冻结 test、重新生成 train/val

`--freeze-test` 保持 `test.jsonl` 不动，并把其中的视频排除出重抽样（`--test-size` 被忽略）。
注意：即使 seed 相同，冻结 test 的运行也不会复现上一轮的 train/val（候选池已变化）。

In [ ]:
# 按需执行（第二轮扩大数据集时）；输出重定向到日志，只看结尾摘要
!mkdir -p log
!.venv/bin/python prepare_data.py --train-size 80 --val-size 100 --freeze-test --probe-content-filter \
    > log/prepare_data.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/prepare_data.log

## 4.（可选）审计已有 split 的内容安全过滤情况（Workflow 步骤 2）

In [ ]:
# 仅生成报告（data/content_filter_probe.json）；输出重定向到日志
!mkdir -p log
!.venv/bin/python probe_content_filter.py \
    > log/probe_content_filter.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/probe_content_filter.log

In [ ]:
# 报告 + 删除被拦截的任务（不回填）；输出重定向到日志
!mkdir -p log
!.venv/bin/python probe_content_filter.py --apply \
    > log/probe_content_filter.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/probe_content_filter.log

In [ ]:
# 基于已有报告重新应用；输出重定向到日志
!mkdir -p log
!.venv/bin/python probe_content_filter.py --from-report \
    > log/probe_content_filter.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/probe_content_filter.log

## 5. 用基线 prompt 调试单条 rollout（Workflow 步骤 3）

In [ ]:
# 单条 rollout 调试；完整日志在 log/frame_agent_debug.log，页面显示尾部（含模型输出与 reward）
!mkdir -p log
!.venv/bin/python frame_agent.py --limit 1 \
    > log/frame_agent_debug.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -40 log/frame_agent_debug.log

## 6. 端到端冒烟测试 APO 循环（Workflow 步骤 4）

最小 beam（1/1/1），成本低，用于验证链路。注意：冒烟运行也会写自己的 `results/<run_id>/`，
并把 `results/latest` 指向它 —— 正式评估前务必确认 latest 指向的是全量运行。

In [ ]:
# 冒烟测试；apo_train 自己会写 log/apo_<run_id>.log，这里再把控制台输出也收进文件。
# 想实时看进度：在终端 tail -f log/apo_smoke_console.log
!mkdir -p log
!.venv/bin/python apo_train.py --smoke \
    > log/apo_smoke_console.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/apo_smoke_console.log

## 7. 全量 APO 运行（Workflow 步骤 5）

每次运行获得一个带时间戳的 run ID：

- APO 日志写入 `log/apo_<run_id>.log`，所有产物写入 `results/<run_id>/`；
- 产物包括：最优 prompt（`best_prompt.txt`）、完整优化报告（每轮候选 prompt、reward、
  gradient 批评、验证得分）`report.md` + `report.json`、精简版本树 `tree.md`、
  以及记录运行参数和 `data/` split 指纹（行数 + 哈希）的 `summary.json`；
- 当最优 prompt 优于种子时，`diffs.md` 展示其派生链每一步的 unified diff（如 v0 → v4 → v7）
  以及整体 seed → best 的 diff；
- 运行之间不会互相覆盖；`results/latest` 指向最新一次运行。

AgentOps SaaS 上传**默认关闭**（span 仍会本地追踪，APO 只需要这个）。
只有需要在 app.agentops.ai 回放会话时才加 `--enable-agentops-service`。

平台说明：Linux + Python ≤ 3.13 会自动并行（默认 `--n-runners 4`，可调）；
macOS / Windows 自动回退到串行 shm 模式（`n_runners=1`），命令相同、只是串行。
大规模运行请在 Linux 上执行；beam 超参与并发调优见 `doc/performance-tuning.md`。

In [ ]:
# 全量 APO（耗时数小时）；控制台输出重定向到文件，结束前页面不会有输出。
# 实时进度：在终端 tail -f log/apo_train_console.log（或 log/apo_<run_id>.log）
!mkdir -p log
!.venv/bin/python apo_train.py \
    > log/apo_train_console.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/apo_train_console.log

### 7b.（可选）从任意历史运行的日志重新生成报告

把下面的 `<run_id>` 替换成实际的 run ID。

In [ ]:
# 注意：apo_train.py 结束时会自动生成报告，此 cell 仅用于事后重新生成。
# 不指定 --log 时自动选最新的 log/apo_<run_id>.log；要指定某次运行，
# 把 RUN_ID 换成实际值（格式如 20260717_020449，可从 ls log/ 或 results/latest/summary.json 查）。
RUN_ID = "<run_id>"  # TODO: 替换为实际 run ID
!.venv/bin/python generate_report.py --log log/apo_{RUN_ID}.log --output-dir results/{RUN_ID} \
    > log/generate_report.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -20 log/generate_report.log

### 7c.（可选）仅从已有 report.md 构建版本树

不需要日志（例如从其他机器拷来的 `report.md`）。此模式下 beam 存活标记不可用。

In [ ]:
!.venv/bin/python generate_report.py --from-report results/latest/report.md \
    > log/generate_report.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -20 log/generate_report.log

## 8. 在保留 test split 上对比基线与调优 prompt（Workflow 步骤 6）

**运行 test 前逐项确认以下前置条件：**

1. **APO 确实产出了优于种子的 prompt** —— 检查 `results/<run_id>/report.md` 或日志结尾；
   若最优 prompt 仍是 v0（`Best prompt not updated`），跑 test 没有意义，先修数据/评估配置并重跑 APO。
2. **确认要评估的 `best_prompt.txt` 来自全量运行而非冒烟运行** —— 检查该运行
   `summary.json` 中的 beam 参数（冒烟是 1/1/1）。
3. **test 必须保持未见** —— 优化期间绝不反复探测 test 分数；所有调优和 prompt 选择只用 val。
   每次基于 test 做的决策都会折损最终数字的可信度。在一轮优化收敛后只跑一次。

两次运行分别写出 `results/eval_baseline.json` 和 `results/eval_tuned.json`
（mean_reward + 每任务明细），对比 `mean_reward` 即最终结论。
若差距小于 val 上观察到的评估噪声（试点配置下约 ±0.015），不要声称有提升 ——
先按 `doc/dataset-sizing.md` 扩大 split 再复核。

In [ ]:
!.venv/bin/python evaluate.py --name baseline \
    > log/eval_baseline.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -20 log/eval_baseline.log

In [ ]:
!.venv/bin/python evaluate.py --prompt results/latest/best_prompt.txt --name tuned \
    > log/eval_tuned.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -20 log/eval_tuned.log

In [ ]:
# 对比两次评估的 mean_reward
import json
from pathlib import Path

for name in ("baseline", "tuned"):
    p = Path(f"results/eval_{name}.json")
    if p.exists():
        data = json.loads(p.read_text())
        print(f"{name:8s} mean_reward = {data.get('mean_reward')}")
    else:
        print(f"{name:8s} 未找到 {p}")

## 附：Reward 定义（v1）

每条 rollout 用 `[0, 1]` 区间的混合 reward 打分：

- `0.2` × `scene_type` 精确匹配
- `0.2` × `is_courier_action` 精确匹配
- `0.6` × LLM judge 对 `english_detail` / `brief` / `title` 的语义评分
- 输出不是合法 JSON 对象 → 得 0 分
- 被 Azure OpenAI 内容安全过滤器拒绝的请求 → 得 0 分（拒绝只取决于输入帧，
  对所有候选 prompt 相同）。默认 94 任务样本探测约 3% 被拦截（不只 `ucf_crime`，
  部分 `Charades` 视频也会触发）。用 `prepare_data.py --probe-content-filter`
  预先排除被拦截视频，或用 `probe_content_filter.py` 审计已有 split。

设计依据与待与客户确认的假设见 `doc/reward-design.md`。

## 附：APO 元提示词

APO 由两个元提示词驱动：*text gradient* 模板（从 rollout trace 批评当前 prompt）和
*apply edit* 模板（重写 prompt）。`apo_train.py` **默认**使用 `prompts/` 下的项目定制版 ——
它们告诉优化器 reward 结构（5 字段 JSON 契约、0.2/0.2/0.6 权重、内容过滤拒绝不是 prompt 的错），
并禁止重写时添加帧/`<video>` 占位符。传 `--default-poml` 可回退到框架内置模板。
详见 `doc/apo-poml-customization.md`。

## 附：冒烟测试

### 离线（无网络、无凭据）

In [ ]:
!mkdir -p log
!.venv/bin/pytest tests/ -v > log/pytest.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -20 log/pytest.log

### 在线（需要 blob 访问 + Azure OpenAI）

In [ ]:
!mkdir -p log
!.venv/bin/python prepare_data.py --train-size 2 --val-size 2 --test-size 2 \
    > log/smoke_online.log 2>&1 && echo "== prepare_data OK ==" || echo "== prepare_data FAILED =="
!.venv/bin/python frame_agent.py --limit 1 \
    >> log/smoke_online.log 2>&1 && echo "== frame_agent OK ==" || echo "== frame_agent FAILED =="
!.venv/bin/python apo_train.py --smoke \
    >> log/smoke_online.log 2>&1 && echo "== apo_train --smoke OK ==" || echo "== apo_train --smoke FAILED =="
!tail -30 log/smoke_online.log